# Week 08 — AI Agent + MCP Shell Server

This notebook connects an OpenAI agent to the `mcp_server` C binary from `week08/aishell/`.
Natural-language queries are routed through the agent → MCP tool call → C shell → JSON result.

```
User NL → OpenAI Agent → MCP tool call → mcp_server (C) → aishell command → JSON → Agent → Answer
```

**Prerequisites**
1. Build the C server: `cd ../aishell && make`
2. Start it: `./mcp_server &` (listens on port 9000)
3. Set your API key: `export OPENAI_API_KEY=sk-...`
4. Install deps: `pip install openai`

## Slide 1 — Architecture
```
┌─────────────┐   NL query    ┌──────────────────┐
│    User     │ ────────────► │   OpenAI Agent   │
└─────────────┘               │  (tool_choice)   │
                              └────────┬─────────┘
                                       │ JSON tool call
                              ┌────────▼─────────┐
                              │  mcp_client.py   │  (this notebook)
                              │  TCP port 9000   │
                              └────────┬─────────┘
                                       │ TCP socket
                              ┌────────▼─────────┐
                              │  mcp_server (C)  │
                              │  fork-per-client │
                              └────────┬─────────┘
                                       │ dispatch
                              ┌────────▼─────────┐
                              │  aishell cmds    │
                              │ ls/echo/pwd/...  │
                              └──────────────────┘
```

## Slide 2 — MCP Client (low-level TCP)

In [ ]:
import socket
import json
import os

MCP_HOST = "127.0.0.1"
MCP_PORT = 9000
TIMEOUT  = 10


def mcp_call(tool: str, args: list[str] = None) -> dict:
    """Send one tool call to the C MCP server and return the parsed JSON response."""
    request = json.dumps({"tool": tool, "args": args or []}) + "\n"

    with socket.create_connection((MCP_HOST, MCP_PORT), timeout=TIMEOUT) as sock:
        sock.sendall(request.encode())
        # Read until newline (server closes after one response)
        buf = b""
        while True:
            chunk = sock.recv(4096)
            if not chunk:
                break
            buf += chunk
            if b"\n" in buf:
                break

    return json.loads(buf.decode())


# Quick smoke test
result = mcp_call("echo", ["MCP", "server", "is", "alive"])
print("exit_code:", result["exit_code"])
print("output   :", result["output"])

## Slide 3 — Discover Available Tools

In [ ]:
import logging

# Set up call log
logging.basicConfig(level=logging.INFO, format="%(asctime)s  %(message)s")
log = logging.getLogger("mcp_agent")


def mcp_call_logged(tool: str, args: list[str] = None) -> dict:
    """Wrapper that logs every agent-to-MCP call (Assignment requirement)."""
    log.info("CALL  tool=%s  args=%s", tool, args)
    result = mcp_call(tool, args)
    log.info("RESULT exit=%d  output=%r", result.get("exit_code", -1),
             result.get("output", result.get("error", ""))[:120])
    return result


# List all registered shell commands
tool_list = mcp_call_logged("__list__")["tools"]
print(f"\n{len(tool_list)} tools available:")
for t in tool_list:
    print(f"  {t['name']:<20} {t['summary']}")

## Slide 4 — Build OpenAI Tool Schemas from MCP Tool List

In [ ]:
def build_openai_tools(tool_list: list[dict]) -> list[dict]:
    """Convert MCP tool list into OpenAI function-calling schemas."""
    schemas = []
    for t in tool_list:
        schemas.append({
            "type": "function",
            "function": {
                "name": t["name"].replace("-", "_"),  # OpenAI names must be [a-z0-9_]
                "description": t["summary"],
                "parameters": {
                    "type": "object",
                    "properties": {
                        "args": {
                            "type": "array",
                            "items": {"type": "string"},
                            "description": "Command-line arguments passed to the shell command"
                        }
                    },
                    "required": []
                }
            }
        })
    return schemas


openai_tools = build_openai_tools(tool_list)
print(f"Built {len(openai_tools)} OpenAI tool schemas")

## Slide 5 — Agent Loop

In [ ]:
from openai import OpenAI

client = OpenAI()  # reads OPENAI_API_KEY from env

# Map sanitised OpenAI name back to MCP tool name
name_map = {t["name"].replace("-", "_"): t["name"] for t in tool_list}


def run_agent(user_query: str, model: str = "gpt-4o-mini") -> str:
    """Run the agent loop: NL → tool calls → MCP → final answer."""
    messages = [
        {"role": "system",
         "content": (
             "You are a Linux shell assistant. Use the provided tools to answer "
             "questions about the local system. Always call a tool to get real data "
             "rather than guessing. Return a concise, human-friendly answer."
         )},
        {"role": "user", "content": user_query}
    ]

    while True:
        resp = client.chat.completions.create(
            model=model,
            messages=messages,
            tools=openai_tools,
            tool_choice="auto"
        )
        msg = resp.choices[0].message
        messages.append(msg)

        if not msg.tool_calls:
            # No more tool calls — agent has a final answer
            return msg.content

        # Execute each tool call via MCP
        for tc in msg.tool_calls:
            fn_name  = tc.function.name
            fn_args  = json.loads(tc.function.arguments)
            mcp_name = name_map.get(fn_name, fn_name)
            args     = fn_args.get("args", [])

            result = mcp_call_logged(mcp_name, args)
            tool_output = result.get("output", result.get("error", ""))

            messages.append({
                "role": "tool",
                "tool_call_id": tc.id,
                "content": tool_output
            })

## Slide 6 — Demo: 3 Natural-Language Queries

Assignment requirement: demonstrate with at least 3 NL queries.

In [ ]:
# Query 1: system identity
q1 = "What operating system and machine architecture am I running on?"
print(f"Q: {q1}")
print(f"A: {run_agent(q1)}\n")

In [ ]:
# Query 2: filesystem
q2 = "List the files in /tmp and tell me how many there are."
print(f"Q: {q2}")
print(f"A: {run_agent(q2)}\n")

In [ ]:
# Query 3: current user & working directory
q3 = "Who am I logged in as, and what is my current working directory?"
print(f"Q: {q3}")
print(f"A: {run_agent(q3)}\n")

## Slide 7 — delete_older_than_days Tool (Assignment Extension)

The assignment asks to add `delete_older_than_days` support. We implement it as a
Python-side tool that wraps `find` + `rm` via MCP (safe: dry-run by default).

In [ ]:
def delete_older_than_days(directory: str, days: int, dry_run: bool = True) -> dict:
    """
    List (and optionally delete) files older than `days` in `directory`.
    dry_run=True (default) only lists; set False to actually remove.
    """
    # Step 1: find files older than N days via 'ls' + filtering
    # We use the shell's 'ls' to list, then filter in Python
    # (A real implementation would call find(1); here we demonstrate the MCP round-trip)
    result = mcp_call_logged("ls", [directory])
    if result["exit_code"] != 0:
        return {"error": result.get("output", "ls failed")}

    import time, os
    now = time.time()
    cutoff = now - days * 86400
    candidates = []

    for name in result["output"].splitlines():
        path = os.path.join(directory, name)
        try:
            mtime = os.path.getmtime(path)
            if mtime < cutoff:
                candidates.append(path)
        except OSError:
            pass

    if dry_run:
        return {"dry_run": True, "would_delete": candidates, "count": len(candidates)}

    deleted = []
    for path in candidates:
        r = mcp_call_logged("rm", [path])
        if r["exit_code"] == 0:
            deleted.append(path)

    return {"dry_run": False, "deleted": deleted, "count": len(deleted)}


# Demo: dry-run on /tmp for files older than 30 days
result = delete_older_than_days("/tmp", days=30, dry_run=True)
print(f"Files in /tmp older than 30 days: {result['count']}")
for p in result["would_delete"][:5]:
    print(f"  {p}")

## Slide 8 — Verification Checklist

| Check | How to verify |
|-------|---------------|
| MCP server listens on 9000 | `ss -tlnp \| grep 9000` |
| Tool list endpoint works | `echo '{"tool":"__list__"}' \| nc 127.0.0.1 9000` |
| echo tool round-trips | Cell 3 above — exit_code 0, correct output |
| ls tool works | Cell 3 above |
| Agent answers Q1 (uname) | Cell in Slide 6 |
| Agent answers Q2 (ls /tmp) | Cell in Slide 6 |
| Agent answers Q3 (whoami/pwd) | Cell in Slide 6 |
| All MCP calls are logged | Check logging output above each answer |
| delete_older_than_days | Slide 7 cell — dry_run=True safe |
| Timeout: server not reachable | Change MCP_PORT to 9001 — `ConnectionRefusedError` |
| net-get command in aishell | `./aishell net-get example.com` |